<table align="left">
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/glasslego/ml-deep-learning-study/blob/main/src/deep_learning_example/02_house_price_prediction.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />구글 코랩에서 실행하기</a>
  </td>
</table>

# Kaggle 집값 예측 - 딥러닝 회귀 문제

이 노트북은 Kaggle의 "House Prices - Advanced Regression Techniques" 대회를 딥러닝으로 해결하는 예제입니다.

## 🎯 목표
- 아이오와 주 에임스의 주택 특성을 바탕으로 집값 예측
- 딥러닝을 활용한 회귀 문제 해결
- 79개의 다양한 특성을 활용한 복잡한 데이터 처리

## 📊 데이터셋 정보
- **train.csv**: 1460개 주택 정보 (집값 포함)
- **test.csv**: 1459개 주택 정보 (예측 대상)
- **특성**: 집 크기, 지역, 건축 연도, 품질 등 79개 특성

In [ ]:
# 필요한 라이브러리 import (Google Colab T4 환경)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder, RobustScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# 나눔고딕 폰트 설치
!sudo apt-get install -y fonts-nanum
!sudo fc-cache -fv
!rm ~/.cache/matplotlib -rf

# 폰트 설정
plt.rc('font', family='NanumGothic')
plt.rc('axes', unicode_minus=False)  # 마이너스 기호 깨짐 방지

print(f"PyTorch 버전: {torch.__version__}")

# GPU 설정 (Google Colab T4 환경)
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Using device: {device}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA 버전: {torch.version.cuda}")
else:
    device = torch.device("cpu")
    print(f"Using device: {device}")

# 시드 설정 (재현 가능한 결과를 위해)
torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

# 시각화 설정
plt.style.use('default')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

## 1. 샘플 데이터 생성 (Kaggle 데이터가 없는 경우)

In [ ]:
def generate_house_price_data():
    """
    Kaggle House Prices 데이터와 유사한 샘플 데이터를 생성합니다.
    """
    np.random.seed(42)
    
    # 훈련 데이터 (1460개)
    n_train = 1460
    
    # 기본 특성 생성
    data = {}
    
    # 1. 연속형 변수들
    data['LotArea'] = np.random.lognormal(9.4, 0.6, n_train).astype(int)  # 부지 면적
    data['GrLivArea'] = np.random.normal(1500, 500, n_train).astype(int)  # 지상 거주 면적
    data['TotalBsmtSF'] = np.random.normal(1000, 400, n_train).astype(int)  # 지하실 면적
    data['1stFlrSF'] = np.random.normal(800, 300, n_train).astype(int)  # 1층 면적
    data['2ndFlrSF'] = np.random.exponential(200, n_train).astype(int)  # 2층 면적
    data['GarageArea'] = np.random.normal(470, 200, n_train).astype(int)  # 차고 면적
    
    # 음수값 제거
    for col in ['LotArea', 'GrLivArea', 'TotalBsmtSF', '1stFlrSF', '2ndFlrSF', 'GarageArea']:
        data[col] = np.maximum(data[col], 0)
    
    # 2. 건축 연도
    data['YearBuilt'] = np.random.choice(range(1872, 2010), n_train, 
                                        p=np.exp(np.linspace(-3, 0, 2010-1872)) / 
                                        np.sum(np.exp(np.linspace(-3, 0, 2010-1872))))
    
    # 3. 품질 등급 (1-10)
    data['OverallQual'] = np.random.choice(range(1, 11), n_train, 
                                          p=[0.02, 0.03, 0.08, 0.12, 0.15, 0.18, 0.16, 0.13, 0.08, 0.05])
    data['OverallCond'] = np.random.choice(range(1, 11), n_train,
                                          p=[0.01, 0.02, 0.06, 0.09, 0.25, 0.28, 0.15, 0.09, 0.04, 0.01])
    
    # 4. 방 개수
    data['BedroomAbvGr'] = np.random.choice(range(0, 9), n_train,
                                           p=[0.02, 0.05, 0.25, 0.45, 0.18, 0.04, 0.01, 0.005, 0.005])
    data['TotRmsAbvGrd'] = data['BedroomAbvGr'] + np.random.choice(range(2, 8), n_train)
    
    # 5. 범주형 변수들
    neighborhoods = ['CollgCr', 'Veenker', 'Crawfor', 'NoRidge', 'Mitchel', 
                    'Somerst', 'NWAmes', 'OldTown', 'BrkSide', 'Sawyer', 
                    'NridgHt', 'NAmes', 'SawyerW', 'IDOTRR', 'MeadowV']
    data['Neighborhood'] = np.random.choice(neighborhoods, n_train)
    
    house_styles = ['1Story', '2Story', '1.5Fin', '1.5Unf', 'SFoyer', 'SLvl', '2.5Unf', '2.5Fin']
    data['HouseStyle'] = np.random.choice(house_styles, n_train, 
                                         p=[0.45, 0.35, 0.08, 0.05, 0.03, 0.02, 0.01, 0.01])
    
    exterior_materials = ['VinylSd', 'MetalSd', 'HdBoard', 'Wd Sdng', 'Plywood', 'WdShing']
    data['Exterior1st'] = np.random.choice(exterior_materials, n_train)
    
    foundations = ['PConc', 'CBlock', 'BrkTil', 'Wood', 'Slab', 'Stone']
    data['Foundation'] = np.random.choice(foundations, n_train,
                                         p=[0.65, 0.20, 0.08, 0.03, 0.02, 0.02])
    
    # 6. 추가 특성들
    data['MSSubClass'] = np.random.choice([20, 30, 40, 50, 60, 70, 75, 80, 85, 90], n_train)
    data['LotFrontage'] = data['LotArea'] / np.random.uniform(8, 20, n_train)
    data['MasVnrArea'] = np.random.exponential(100, n_train)
    
    # 결측치 추가
    missing_indices = np.random.choice(n_train, int(n_train * 0.15), replace=False)
    data['LotFrontage'][missing_indices] = np.nan
    
    missing_indices = np.random.choice(n_train, int(n_train * 0.08), replace=False)
    data['MasVnrArea'][missing_indices] = np.nan
    
    # 집값 생성 (복잡한 공식 사용)
    price_base = 50000  # 기본 가격
    
    # 각 요소의 가격 기여도
    price = price_base + \
            data['GrLivArea'] * 80 + \
            data['TotalBsmtSF'] * 40 + \
            data['GarageArea'] * 60 + \
            data['OverallQual'] * 15000 + \
            data['OverallCond'] * 5000 + \
            (2010 - data['YearBuilt']) * (-200) + \
            data['TotRmsAbvGrd'] * 3000
    
    # 동네별 가격 조정
    neighborhood_multiplier = {
        'NoRidge': 1.4, 'NridgHt': 1.3, 'Somerst': 1.2, 'Veenker': 1.15,
        'Crawfor': 1.1, 'CollgCr': 1.05, 'NWAmes': 1.0, 'Mitchel': 0.95,
        'NAmes': 0.9, 'Sawyer': 0.85, 'SawyerW': 0.85, 'OldTown': 0.8,
        'BrkSide': 0.75, 'IDOTRR': 0.7, 'MeadowV': 0.65
    }
    
    for i in range(n_train):
        price[i] *= neighborhood_multiplier.get(data['Neighborhood'][i], 1.0)
    
    # 노이즈 추가
    price *= np.random.normal(1, 0.15, n_train)
    price = np.maximum(price, 30000).astype(int)  # 최소 가격 설정
    
    data['SalePrice'] = price
    
    # DataFrame 생성
    train_df = pd.DataFrame(data)
    train_df['Id'] = range(1, n_train + 1)
    
    # 테스트 데이터 생성 (SalePrice 제외)
    n_test = 1459
    test_data = {}
    
    # 테스트 데이터도 비슷한 분포로 생성
    test_data['LotArea'] = np.random.lognormal(9.4, 0.6, n_test).astype(int)
    test_data['GrLivArea'] = np.random.normal(1500, 500, n_test).astype(int)
    test_data['TotalBsmtSF'] = np.random.normal(1000, 400, n_test).astype(int)
    test_data['1stFlrSF'] = np.random.normal(800, 300, n_test).astype(int)
    test_data['2ndFlrSF'] = np.random.exponential(200, n_test).astype(int)
    test_data['GarageArea'] = np.random.normal(470, 200, n_test).astype(int)
    
    for col in ['LotArea', 'GrLivArea', 'TotalBsmtSF', '1stFlrSF', '2ndFlrSF', 'GarageArea']:
        test_data[col] = np.maximum(test_data[col], 0)
    
    test_data['YearBuilt'] = np.random.choice(range(1872, 2010), n_test,
                                             p=np.exp(np.linspace(-3, 0, 2010-1872)) / 
                                             np.sum(np.exp(np.linspace(-3, 0, 2010-1872))))
    
    test_data['OverallQual'] = np.random.choice(range(1, 11), n_test,
                                               p=[0.02, 0.03, 0.08, 0.12, 0.15, 0.18, 0.16, 0.13, 0.08, 0.05])
    test_data['OverallCond'] = np.random.choice(range(1, 11), n_test,
                                               p=[0.01, 0.02, 0.06, 0.09, 0.25, 0.28, 0.15, 0.09, 0.04, 0.01])
    
    test_data['BedroomAbvGr'] = np.random.choice(range(0, 9), n_test,
                                                p=[0.02, 0.05, 0.25, 0.45, 0.18, 0.04, 0.01, 0.005, 0.005])
    test_data['TotRmsAbvGrd'] = test_data['BedroomAbvGr'] + np.random.choice(range(2, 8), n_test)
    
    test_data['Neighborhood'] = np.random.choice(neighborhoods, n_test)
    test_data['HouseStyle'] = np.random.choice(house_styles, n_test,
                                              p=[0.45, 0.35, 0.08, 0.05, 0.03, 0.02, 0.01, 0.01])
    test_data['Exterior1st'] = np.random.choice(exterior_materials, n_test)
    test_data['Foundation'] = np.random.choice(foundations, n_test,
                                              p=[0.65, 0.20, 0.08, 0.03, 0.02, 0.02])
    
    test_data['MSSubClass'] = np.random.choice([20, 30, 40, 50, 60, 70, 75, 80, 85, 90], n_test)
    test_data['LotFrontage'] = test_data['LotArea'] / np.random.uniform(8, 20, n_test)
    test_data['MasVnrArea'] = np.random.exponential(100, n_test)
    
    # 테스트 데이터에도 결측치 추가
    missing_indices = np.random.choice(n_test, int(n_test * 0.15), replace=False)
    test_data['LotFrontage'][missing_indices] = np.nan
    
    missing_indices = np.random.choice(n_test, int(n_test * 0.08), replace=False)
    test_data['MasVnrArea'][missing_indices] = np.nan
    
    test_df = pd.DataFrame(test_data)
    test_df['Id'] = range(n_train + 1, n_train + n_test + 1)
    
    return train_df, test_df

# 데이터 로드 또는 생성
try:
    # 실제 Kaggle 데이터 시도
    train_df = pd.read_csv('train.csv')
    test_df = pd.read_csv('test.csv')
    print("✅ Kaggle House Prices 데이터를 성공적으로 로드했습니다!")
except FileNotFoundError:
    # 샘플 데이터 생성
    print("📊 Kaggle 데이터가 없어 샘플 데이터를 생성합니다...")
    train_df, test_df = generate_house_price_data()
    print("✅ 샘플 House Prices 데이터를 생성했습니다!")

print(f"\n📈 데이터 크기:")
print(f"- 훈련 데이터: {train_df.shape}")
print(f"- 테스트 데이터: {test_df.shape}")

# 기본 정보 출력
print(f"\n💰 집값 기본 통계:")
print(f"- 평균: ${train_df['SalePrice'].mean():,.0f}")
print(f"- 중앙값: ${train_df['SalePrice'].median():,.0f}")
print(f"- 최소값: ${train_df['SalePrice'].min():,.0f}")
print(f"- 최대값: ${train_df['SalePrice'].max():,.0f}")

## 2. 데이터 탐색 (EDA)

In [ ]:
# 데이터 정보 확인
print("🔍 데이터 기본 정보")
print("=" * 50)
print(f"훈련 데이터 형태: {train_df.shape}")
print(f"특성 개수: {train_df.shape[1] - 1}개 (Id, SalePrice 제외)")

# 데이터 타입 확인
numeric_features = train_df.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = train_df.select_dtypes(include=['object']).columns.tolist()

# Id와 SalePrice 제외
if 'Id' in numeric_features:
    numeric_features.remove('Id')
if 'SalePrice' in numeric_features:
    numeric_features.remove('SalePrice')

print(f"\n📊 특성 타입:")
print(f"- 수치형 특성: {len(numeric_features)}개")
print(f"- 범주형 특성: {len(categorical_features)}개")

# 결측치 확인
missing_data = train_df.isnull().sum()
missing_data = missing_data[missing_data > 0].sort_values(ascending=False)

if len(missing_data) > 0:
    print(f"\n🔍 결측치 현황:")
    for col, count in missing_data.head(10).items():
        percentage = (count / len(train_df)) * 100
        print(f"- {col}: {count}개 ({percentage:.1f}%)")
else:
    print(f"\n✅ 결측치 없음")

In [ ]:
# 집값 분포 시각화
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# 1. 집값 분포
ax1 = axes[0, 0]
ax1.hist(train_df['SalePrice'], bins=50, alpha=0.7, edgecolor='black')
ax1.set_xlabel('Sale Price ($)')
ax1.set_ylabel('Frequency')
ax1.set_title('집값 분포')
ax1.ticklabel_format(style='plain', axis='x')

# 2. 로그 변환된 집값 분포
ax2 = axes[0, 1]
log_prices = np.log1p(train_df['SalePrice'])
ax2.hist(log_prices, bins=50, alpha=0.7, edgecolor='black', color='orange')
ax2.set_xlabel('Log(Sale Price + 1)')
ax2.set_ylabel('Frequency')
ax2.set_title('로그 변환된 집값 분포')

# 3. 거주 면적 vs 집값
ax3 = axes[0, 2]
ax3.scatter(train_df['GrLivArea'], train_df['SalePrice'], alpha=0.6)
ax3.set_xlabel('Ground Living Area (sqft)')
ax3.set_ylabel('Sale Price ($)')
ax3.set_title('거주 면적 vs 집값')

# 4. 전체 품질 vs 집값
ax4 = axes[1, 0]
if 'OverallQual' in train_df.columns:
    quality_price = train_df.groupby('OverallQual')['SalePrice'].mean()
    ax4.bar(quality_price.index, quality_price.values, alpha=0.7)
    ax4.set_xlabel('Overall Quality')
    ax4.set_ylabel('Average Sale Price ($)')
    ax4.set_title('품질별 평균 집값')

# 5. 건축 연도 vs 집값
ax5 = axes[1, 1]
if 'YearBuilt' in train_df.columns:
    ax5.scatter(train_df['YearBuilt'], train_df['SalePrice'], alpha=0.6)
    ax5.set_xlabel('Year Built')
    ax5.set_ylabel('Sale Price ($)')
    ax5.set_title('건축 연도 vs 집값')

# 6. 동네별 평균 집값 (상위 10개)
ax6 = axes[1, 2]
if 'Neighborhood' in train_df.columns:
    neighborhood_price = train_df.groupby('Neighborhood')['SalePrice'].mean().sort_values(ascending=False)
    top_neighborhoods = neighborhood_price.head(10)
    ax6.barh(range(len(top_neighborhoods)), top_neighborhoods.values, alpha=0.7)
    ax6.set_yticks(range(len(top_neighborhoods)))
    ax6.set_yticklabels(top_neighborhoods.index)
    ax6.set_xlabel('Average Sale Price ($)')
    ax6.set_title('동네별 평균 집값 (상위 10개)')
    ax6.invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
# 수치형 특성들과 집값의 상관관계 분석
numeric_features_with_price = numeric_features + ['SalePrice']
correlation_matrix = train_df[numeric_features_with_price].corr()

# 집값과의 상관관계 높은 특성들
price_correlations = correlation_matrix['SalePrice'].abs().sort_values(ascending=False)

print("💡 집값과 상관관계가 높은 특성들 (상위 10개):")
print("-" * 50)
for i, (feature, corr) in enumerate(price_correlations.head(11).items()):
    if feature != 'SalePrice':  # SalePrice 자체 제외
        print(f"{i:2d}. {feature:20s}: {corr:6.3f}")

# 상관관계 히트맵 (상위 15개 특성)
top_features = price_correlations.head(16).index.tolist()  # SalePrice 포함
top_corr_matrix = correlation_matrix.loc[top_features, top_features]

plt.figure(figsize=(12, 10))
mask = np.triu(np.ones_like(top_corr_matrix, dtype=bool))
sns.heatmap(top_corr_matrix, mask=mask, annot=True, cmap='coolwarm', 
            center=0, square=True, linewidths=0.5, fmt='.2f')
plt.title('상위 특성들 간 상관관계')
plt.tight_layout()
plt.show()

## 3. 데이터 전처리

In [ ]:
def preprocess_house_data(train_df, test_df):
    """
    House Prices 데이터를 전처리합니다.
    """
    print("🔧 데이터 전처리 시작...")
    
    # 데이터 복사
    train_processed = train_df.copy()
    test_processed = test_df.copy()
    
    # 타겟 변수 분리 및 로그 변환
    y_train = train_processed['SalePrice'].values
    y_train_log = np.log1p(y_train)  # 로그 변환으로 정규분포에 가깝게
    
    # Id와 SalePrice 제거
    train_processed = train_processed.drop(['Id', 'SalePrice'], axis=1)
    test_processed = test_processed.drop(['Id'], axis=1)
    
    # 전체 데이터 결합 (일관된 전처리를 위해)
    all_data = pd.concat([train_processed, test_processed], ignore_index=True)
    
    print(f"결합된 데이터 크기: {all_data.shape}")
    
    # 1. 결측치 처리
    missing_counts = all_data.isnull().sum()
    missing_features = missing_counts[missing_counts > 0]
    
    if len(missing_features) > 0:
        print(f"\n📋 결측치 처리:")
        for feature in missing_features.index:
            if all_data[feature].dtype == 'object':
                # 범주형 변수: 최빈값으로 채우기
                mode_value = all_data[feature].mode()[0] if len(all_data[feature].mode()) > 0 else 'Unknown'
                all_data[feature].fillna(mode_value, inplace=True)
                print(f"  - {feature}: 최빈값 '{mode_value}'으로 채움")
            else:
                # 수치형 변수: 중앙값으로 채우기
                median_value = all_data[feature].median()
                all_data[feature].fillna(median_value, inplace=True)
                print(f"  - {feature}: 중앙값 {median_value:.1f}으로 채움")
    
    # 2. 새로운 특성 생성 (Feature Engineering)
    print(f"\n🔨 특성 엔지니어링:")
    
    # 전체 면적
    if 'TotalBsmtSF' in all_data.columns and '1stFlrSF' in all_data.columns and '2ndFlrSF' in all_data.columns:
        all_data['TotalSF'] = all_data['TotalBsmtSF'] + all_data['1stFlrSF'] + all_data['2ndFlrSF']
        print("  - TotalSF: 전체 면적 생성")
    
    # 집 나이
    if 'YearBuilt' in all_data.columns:
        all_data['HouseAge'] = 2024 - all_data['YearBuilt']
        print("  - HouseAge: 집 나이 생성")
    
    # 전체 욕실 개수
    bathroom_cols = [col for col in all_data.columns if 'Bath' in col]
    if len(bathroom_cols) > 0:
        all_data['TotalBath'] = 0
        for col in bathroom_cols:
            if all_data[col].dtype in ['int64', 'float64']:
                all_data['TotalBath'] += all_data[col]
        print(f"  - TotalBath: 전체 욕실 개수 ({len(bathroom_cols)}개 컬럼 합계)")
    
    # 3. 범주형 변수 인코딩
    categorical_columns = all_data.select_dtypes(include=['object']).columns
    
    if len(categorical_columns) > 0:
        print(f"\n🏷️ 범주형 변수 인코딩 ({len(categorical_columns)}개):")
        
        # 라벨 인코딩
        label_encoders = {}
        for col in categorical_columns:
            le = LabelEncoder()
            all_data[col] = le.fit_transform(all_data[col].astype(str))
            label_encoders[col] = le
            print(f"  - {col}: {len(le.classes_)}개 카테고리")
    
    # 4. 이상치 제거 (훈련 데이터에서만)
    print(f"\n🎯 이상치 감지:")
    outlier_indices = []
    
    # Z-score를 사용한 이상치 감지 (수치형 특성만)
    numeric_cols = all_data.select_dtypes(include=[np.number]).columns
    train_data = all_data[:len(train_df)]
    
    for col in numeric_cols:
        z_scores = np.abs(stats.zscore(train_data[col]))
        outliers = np.where(z_scores > 3)[0]
        if len(outliers) > 0:
            outlier_indices.extend(outliers)
    
    outlier_indices = list(set(outlier_indices))
    print(f"  - 감지된 이상치: {len(outlier_indices)}개")
    
    # 5. 데이터 분할
    train_size = len(train_df)
    X_train = all_data[:train_size].values
    X_test = all_data[train_size:].values
    
    # 이상치 제거 (옵션)
    if len(outlier_indices) > 0 and len(outlier_indices) < len(X_train) * 0.05:  # 5% 미만일 때만
        print(f"  - 이상치 {len(outlier_indices)}개 제거")
        mask = np.ones(len(X_train), dtype=bool)
        mask[outlier_indices] = False
        X_train = X_train[mask]
        y_train_log = y_train_log[mask]
        y_train = y_train[mask]
    
    # 6. 스케일링
    scaler = RobustScaler()  # 이상치에 강한 스케일러
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    print(f"\n✅ 전처리 완료!")
    print(f"- 훈련 데이터: {X_train_scaled.shape}")
    print(f"- 테스트 데이터: {X_test_scaled.shape}")
    print(f"- 특성 개수: {X_train_scaled.shape[1]}")
    
    return X_train_scaled, X_test_scaled, y_train, y_train_log, scaler, all_data.columns.tolist()

# 데이터 전처리 실행
X_train, X_test, y_train, y_train_log, scaler, feature_names = preprocess_house_data(train_df, test_df)

print(f"\n📋 처리된 특성들 ({len(feature_names)}개):")
for i, feature in enumerate(feature_names[:20]):  # 처음 20개만 출력
    print(f"{i+1:2d}. {feature}")
if len(feature_names) > 20:
    print(f"   ... 및 {len(feature_names) - 20}개 추가 특성")

## 4. 딥러닝 모델 구현

In [ ]:
class HousePriceNet(nn.Module):
    """
    집값 예측을 위한 딥러닝 회귀 모델
    """
    def __init__(self, input_size, hidden_sizes=[256, 128, 64, 32], dropout_rate=0.3):
        super(HousePriceNet, self).__init__()
        
        layers = []
        prev_size = input_size
        
        # 은닉층들 구성
        for i, hidden_size in enumerate(hidden_sizes):
            layers.extend([
                nn.Linear(prev_size, hidden_size),
                nn.BatchNorm1d(hidden_size),
                nn.ReLU(),
                nn.Dropout(dropout_rate)
            ])
            prev_size = hidden_size
        
        # 출력층 (회귀이므로 활성화 함수 없음)
        layers.append(nn.Linear(prev_size, 1))
        
        self.network = nn.Sequential(*layers)
        
        # 가중치 초기화
        self.apply(self._init_weights)
    
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.xavier_uniform_(module.weight)
            nn.init.constant_(module.bias, 0)
    
    def forward(self, x):
        return self.network(x)

# 다양한 모델 구성 정의
model_configs = {
    'Simple': {
        'hidden_sizes': [128, 64], 
        'dropout_rate': 0.2
    },
    'Deep': {
        'hidden_sizes': [512, 256, 128, 64, 32], 
        'dropout_rate': 0.4
    },
    'Wide': {
        'hidden_sizes': [512, 256], 
        'dropout_rate': 0.3
    },
    'Balanced': {
        'hidden_sizes': [256, 128, 64, 32], 
        'dropout_rate': 0.3
    },
    'Complex': {
        'hidden_sizes': [512, 256, 128, 64, 32, 16], 
        'dropout_rate': 0.4
    }
}

# 모델들 생성
input_size = X_train.shape[1]
models = {}

print(f"🏗️ 모델 생성 (입력 크기: {input_size})")
print("=" * 60)

for name, config in model_configs.items():
    model = HousePriceNet(input_size, **config).to(device)
    models[name] = model
    
    # 파라미터 수 계산
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    print(f"{name:>10s}: {total_params:>8,} 파라미터 (훈련 가능: {trainable_params:>8,})")

# 대표 모델 구조 출력
print(f"\n🏗️ Balanced 모델 구조:")
print(models['Balanced'])

## 5. 모델 학습 및 평가

In [ ]:
def train_regression_model(model, X_train, y_train, X_val, y_val,
                          epochs=200, batch_size=64, lr=0.001, patience=20):
    """
    회귀 모델을 훈련시키고 결과를 반환합니다.
    """
    # 데이터 로더 준비
    train_dataset = TensorDataset(
        torch.FloatTensor(X_train),
        torch.FloatTensor(y_train).unsqueeze(1)
    )
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    
    val_dataset = TensorDataset(
        torch.FloatTensor(X_val),
        torch.FloatTensor(y_val).unsqueeze(1)
    )
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    
    # 최적화 설정
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', patience=10, factor=0.5, verbose=False
    )
    
    # 학습 기록
    history = {
        'train_loss': [],
        'val_loss': [],
        'train_rmse': [],
        'val_rmse': []
    }
    
    best_val_loss = float('inf')
    patience_counter = 0
    best_model_state = None
    
    for epoch in range(epochs):
        # 훈련 모드
        model.train()
        train_loss = 0
        train_predictions = []
        train_targets = []
        
        for batch_X, batch_y in train_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            
            # 그래디언트 클리핑
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            optimizer.step()
            
            train_loss += loss.item()
            train_predictions.extend(outputs.detach().cpu().numpy().flatten())
            train_targets.extend(batch_y.detach().cpu().numpy().flatten())
        
        # 검증 모드
        model.eval()
        val_loss = 0
        val_predictions = []
        val_targets = []
        
        with torch.no_grad():
            for batch_X, batch_y in val_loader:
                batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                
                outputs = model(batch_X)
                loss = criterion(outputs, batch_y)
                
                val_loss += loss.item()
                val_predictions.extend(outputs.cpu().numpy().flatten())
                val_targets.extend(batch_y.cpu().numpy().flatten())
        
        # 평균 손실 및 RMSE 계산
        avg_train_loss = train_loss / len(train_loader)
        avg_val_loss = val_loss / len(val_loader)
        
        train_rmse = np.sqrt(mean_squared_error(train_targets, train_predictions))
        val_rmse = np.sqrt(mean_squared_error(val_targets, val_predictions))
        
        # 기록
        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(avg_val_loss)
        history['train_rmse'].append(train_rmse)
        history['val_rmse'].append(val_rmse)
        
        # 스케줄러 업데이트
        scheduler.step(avg_val_loss)
        
        # 조기 종료 체크
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            patience_counter = 0
            best_model_state = model.state_dict().copy()
        else:
            patience_counter += 1
        
        # 진행 상황 출력
        if epoch % 40 == 0 or epoch == epochs - 1:
            print(f'Epoch {epoch:3d}: Train Loss={avg_train_loss:.4f}, '
                  f'Val Loss={avg_val_loss:.4f}, Train RMSE={train_rmse:.4f}, '
                  f'Val RMSE={val_rmse:.4f}')
        
        if patience_counter >= patience:
            print(f'조기 종료: {epoch+1} 에폭에서 훈련 종료')
            break
    
    # 최고 모델 복원
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
    
    return history

# 데이터 분할 (훈련/검증)
X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(
    X_train, y_train_log, test_size=0.2, random_state=42
)

print(f"📊 데이터 분할 결과:")
print(f"- 훈련: {X_train_split.shape[0]:,}개")
print(f"- 검증: {X_val_split.shape[0]:,}개")

# 모든 모델 훈련
results = {}
print(f"\n🚀 모델 훈련 시작...")
print("=" * 70)

for name, model in models.items():
    print(f"\n📊 {name} 모델 훈련 중...")
    
    history = train_regression_model(
        model, X_train_split, y_train_split, X_val_split, y_val_split,
        epochs=200, batch_size=64, lr=0.001, patience=25
    )
    
    results[name] = {
        'model': model,
        'history': history,
        'best_val_rmse': min(history['val_rmse'])
    }
    
    print(f"✅ {name} 모델 완료 - 최고 검증 RMSE: {results[name]['best_val_rmse']:.4f}")

print(f"\n🏆 모든 모델 훈련 완료!")

## 6. 결과 시각화 및 분석

In [ ]:
# 학습 결과 시각화
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 1. 손실 함수 비교
ax1 = axes[0, 0]
for name, result in results.items():
    history = result['history']
    ax1.plot(history['train_loss'], label=f'{name} (Train)', linestyle='-', alpha=0.7)
    ax1.plot(history['val_loss'], label=f'{name} (Val)', linestyle='--', alpha=0.7)

ax1.set_xlabel('Epoch')
ax1.set_ylabel('MSE Loss')
ax1.set_title('모델별 손실 함수 변화')
ax1.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
ax1.grid(True, alpha=0.3)
ax1.set_yscale('log')

# 2. RMSE 비교
ax2 = axes[0, 1]
for name, result in results.items():
    history = result['history']
    ax2.plot(history['train_rmse'], label=f'{name} (Train)', linestyle='-', alpha=0.7)
    ax2.plot(history['val_rmse'], label=f'{name} (Val)', linestyle='--', alpha=0.7)

ax2.set_xlabel('Epoch')
ax2.set_ylabel('RMSE')
ax2.set_title('모델별 RMSE 변화')
ax2.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
ax2.grid(True, alpha=0.3)

# 3. 최종 성능 비교
ax3 = axes[1, 0]
model_names = list(results.keys())
best_val_rmses = [results[name]['best_val_rmse'] for name in model_names]
final_train_rmses = [results[name]['history']['train_rmse'][-1] for name in model_names]

x = np.arange(len(model_names))
width = 0.35

bars1 = ax3.bar(x - width/2, final_train_rmses, width, label='Train RMSE', alpha=0.7)
bars2 = ax3.bar(x + width/2, best_val_rmses, width, label='Best Val RMSE', alpha=0.7)

ax3.set_xlabel('Model')
ax3.set_ylabel('RMSE')
ax3.set_title('모델별 최종 성능 비교')
ax3.set_xticks(x)
ax3.set_xticklabels(model_names, rotation=45)
ax3.legend()

# 값 표시
for bar in bars1:
    height = bar.get_height()
    ax3.text(bar.get_x() + bar.get_width()/2., height + 0.005,
             f'{height:.3f}', ha='center', va='bottom', fontsize=9)

for bar in bars2:
    height = bar.get_height()
    ax3.text(bar.get_x() + bar.get_width()/2., height + 0.005,
             f'{height:.3f}', ha='center', va='bottom', fontsize=9)

# 4. 과적합 분석
ax4 = axes[1, 1]
overfitting_scores = []
for name in model_names:
    history = results[name]['history']
    final_train_rmse = history['train_rmse'][-1]
    best_val_rmse = min(history['val_rmse'])
    overfitting = final_train_rmse - best_val_rmse
    overfitting_scores.append(overfitting)

colors = ['green' if score > -0.05 else 'orange' if score > -0.1 else 'red' 
          for score in overfitting_scores]
bars = ax4.bar(model_names, overfitting_scores, color=colors, alpha=0.7)
ax4.axhline(y=0, color='black', linestyle='--', alpha=0.7)
ax4.set_xlabel('Model')
ax4.set_ylabel('Train RMSE - Val RMSE')
ax4.set_title('과적합 분석')
ax4.tick_params(axis='x', rotation=45)

for bar, score in zip(bars, overfitting_scores):
    ax4.text(bar.get_x() + bar.get_width()/2., score + 0.002,
             f'{score:.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

# 최고 성능 모델 선택
best_model_name = min(results.keys(), key=lambda x: results[x]['best_val_rmse'])
best_model = results[best_model_name]['model']
best_rmse = results[best_model_name]['best_val_rmse']

print(f"🏆 최고 성능 모델: {best_model_name}")
print(f"🎯 최고 검증 RMSE: {best_rmse:.4f} (로그 스케일)")

# 달러 단위로 변환 (대략적 추정)
rmse_dollars = np.expm1(best_rmse)  # 로그 역변환
print(f"🎯 달러 단위 추정 RMSE: ${rmse_dollars:,.0f}")

# 모델별 성능 요약
print(f"\n📊 모델별 성능 요약:")
print("-" * 70)
for name in model_names:
    history = results[name]['history']
    train_rmse = history['train_rmse'][-1]
    val_rmse = results[name]['best_val_rmse']
    overfitting = train_rmse - val_rmse
    
    print(f"{name:>10}: Train RMSE {train_rmse:.3f}, Val RMSE {val_rmse:.3f}, "
          f"Overfitting {overfitting:.3f}")

## 7. 예측 성능 상세 분석

In [ ]:
# 최고 성능 모델로 상세 분석
best_model.eval()
X_val_tensor = torch.FloatTensor(X_val_split).to(device)

with torch.no_grad():
    val_predictions_log = best_model(X_val_tensor).cpu().numpy().flatten()

# 로그 스케일을 원래 가격으로 변환
val_predictions = np.expm1(val_predictions_log)
val_true = np.expm1(y_val_split)

# 성능 지표 계산
mse = mean_squared_error(val_true, val_predictions)
rmse = np.sqrt(mse)
mae = mean_absolute_error(val_true, val_predictions)
r2 = r2_score(val_true, val_predictions)

print(f"📊 {best_model_name} 모델 상세 성능 (달러 단위):")
print("=" * 50)
print(f"RMSE:        ${rmse:,.0f}")
print(f"MAE:         ${mae:,.0f}")
print(f"R² Score:    {r2:.4f}")
print(f"평균 집값:   ${np.mean(val_true):,.0f}")
print(f"RMSE/평균:   {rmse/np.mean(val_true)*100:.1f}%")

# 예측 vs 실제 시각화
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# 1. 예측 vs 실제 산점도
ax1 = axes[0, 0]
ax1.scatter(val_true, val_predictions, alpha=0.6)
ax1.plot([val_true.min(), val_true.max()], [val_true.min(), val_true.max()], 
         'r--', linewidth=2, label='Perfect Prediction')
ax1.set_xlabel('실제 집값 ($)')
ax1.set_ylabel('예측 집값 ($)')
ax1.set_title(f'{best_model_name} - 예측 vs 실제')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 2. 잔차 분포
ax2 = axes[0, 1]
residuals = val_predictions - val_true
ax2.hist(residuals, bins=50, alpha=0.7, edgecolor='black')
ax2.axvline(0, color='red', linestyle='--', linewidth=2)
ax2.set_xlabel('잔차 (예측 - 실제) ($)')
ax2.set_ylabel('빈도')
ax2.set_title('잔차 분포')
ax2.grid(True, alpha=0.3)

# 3. 잔차 vs 예측값
ax3 = axes[1, 0]
ax3.scatter(val_predictions, residuals, alpha=0.6)
ax3.axhline(0, color='red', linestyle='--', linewidth=2)
ax3.set_xlabel('예측 집값 ($)')
ax3.set_ylabel('잔차 ($)')
ax3.set_title('잔차 vs 예측값')
ax3.grid(True, alpha=0.3)

# 4. 가격대별 성능
ax4 = axes[1, 1]
# 가격대별 그룹화
price_ranges = [(0, 150000), (150000, 250000), (250000, 400000), (400000, float('inf'))]
range_labels = ['<$150K', '$150K-250K', '$250K-400K', '>$400K']
range_rmses = []

for min_price, max_price in price_ranges:
    mask = (val_true >= min_price) & (val_true < max_price)
    if np.sum(mask) > 0:
        range_rmse = np.sqrt(mean_squared_error(val_true[mask], val_predictions[mask]))
        range_rmses.append(range_rmse)
    else:
        range_rmses.append(0)

bars = ax4.bar(range_labels, range_rmses, alpha=0.7)
ax4.set_xlabel('가격대')
ax4.set_ylabel('RMSE ($)')
ax4.set_title('가격대별 RMSE')
ax4.tick_params(axis='x', rotation=45)

for bar, rmse_val in zip(bars, range_rmses):
    if rmse_val > 0:
        ax4.text(bar.get_x() + bar.get_width()/2., rmse_val + 1000,
                 f'${rmse_val:,.0f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

# 예측 정확도 분석
percentage_errors = np.abs(residuals) / val_true * 100
print(f"\n🎯 예측 정확도 분석:")
print(f"- 10% 이내 정확도: {np.sum(percentage_errors <= 10) / len(percentage_errors) * 100:.1f}%")
print(f"- 20% 이내 정확도: {np.sum(percentage_errors <= 20) / len(percentage_errors) * 100:.1f}%")
print(f"- 평균 절대 오차율: {np.mean(percentage_errors):.1f}%")
print(f"- 중간 절대 오차율: {np.median(percentage_errors):.1f}%")

## 8. 테스트 데이터 예측 및 제출 파일 생성

In [ ]:
# 최고 성능 모델로 테스트 데이터 예측
best_model.eval()
X_test_tensor = torch.FloatTensor(X_test).to(device)

with torch.no_grad():
    test_predictions_log = best_model(X_test_tensor).cpu().numpy().flatten()

# 로그 스케일을 원래 가격으로 변환
test_predictions = np.expm1(test_predictions_log)

# 예측 결과 분석
print(f"🔮 테스트 데이터 예측 결과:")
print(f"- 총 테스트 샘플: {len(test_predictions):,}개")
print(f"- 예측 평균 집값: ${np.mean(test_predictions):,.0f}")
print(f"- 예측 중앙값: ${np.median(test_predictions):,.0f}")
print(f"- 예측 최소값: ${np.min(test_predictions):,.0f}")
print(f"- 예측 최대값: ${np.max(test_predictions):,.0f}")

# 예측 분포 시각화
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# 1. 예측 집값 분포
ax1.hist(test_predictions, bins=50, alpha=0.7, edgecolor='black')
ax1.axvline(np.mean(test_predictions), color='red', linestyle='--', 
           linewidth=2, label=f'평균: ${np.mean(test_predictions):,.0f}')
ax1.axvline(np.median(test_predictions), color='green', linestyle='--', 
           linewidth=2, label=f'중앙값: ${np.median(test_predictions):,.0f}')
ax1.set_xlabel('예측 집값 ($)')
ax1.set_ylabel('빈도')
ax1.set_title('테스트 데이터 예측 집값 분포')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 2. 가격대별 분포
price_bins = [0, 100000, 200000, 300000, 500000, float('inf')]
price_labels = ['<$100K', '$100K-200K', '$200K-300K', '$300K-500K', '>$500K']
price_counts = []

for i in range(len(price_bins)-1):
    count = np.sum((test_predictions >= price_bins[i]) & (test_predictions < price_bins[i+1]))
    price_counts.append(count)

ax2.pie(price_counts, labels=price_labels, autopct='%1.1f%%', startangle=90)
ax2.set_title('가격대별 예측 분포')

plt.tight_layout()
plt.show()

# Kaggle 제출용 파일 생성
submission = pd.DataFrame({
    'Id': test_df['Id'],
    'SalePrice': test_predictions
})

# 파일 저장
submission_filename = f'house_prices_submission_{best_model_name.lower()}.csv'
submission.to_csv(submission_filename, index=False)

print(f"\n📁 제출 파일 생성 완료: {submission_filename}")
print(f"\n📋 제출 파일 미리보기:")
print(submission.head(10))

print(f"\n🎯 Kaggle 제출 방법:")
print(f"1. Kaggle House Prices 대회 페이지 방문")
print(f"2. Submit Predictions 클릭")
print(f"3. {submission_filename} 파일 업로드")
print(f"4. 결과 확인 및 리더보드 순위 체크")

# 예측값 통계
print(f"\n📊 예측값 상세 통계:")
print(f"- 표준편차: ${np.std(test_predictions):,.0f}")
print(f"- 25% 분위수: ${np.percentile(test_predictions, 25):,.0f}")
print(f"- 75% 분위수: ${np.percentile(test_predictions, 75):,.0f}")
print(f"- 변동계수: {np.std(test_predictions)/np.mean(test_predictions)*100:.1f}%")

## 9. 앙상블 모델 (보너스)

In [ ]:
# 모든 모델의 예측을 결합한 앙상블
print("🎭 앙상블 모델 생성...")

# 각 모델의 검증 데이터 예측
ensemble_predictions_val = []
ensemble_predictions_test = []
model_weights = []

X_val_tensor = torch.FloatTensor(X_val_split).to(device)
X_test_tensor = torch.FloatTensor(X_test).to(device)

for name, result in results.items():
    model = result['model']
    model.eval()
    
    with torch.no_grad():
        # 검증 데이터 예측
        val_pred = model(X_val_tensor).cpu().numpy().flatten()
        ensemble_predictions_val.append(val_pred)
        
        # 테스트 데이터 예측
        test_pred = model(X_test_tensor).cpu().numpy().flatten()
        ensemble_predictions_test.append(test_pred)
    
    # 가중치는 성능에 반비례 (RMSE가 낮을수록 높은 가중치)
    weight = 1.0 / result['best_val_rmse']
    model_weights.append(weight)

# 가중치 정규화
model_weights = np.array(model_weights)
model_weights = model_weights / np.sum(model_weights)

print(f"\n📊 앙상블 가중치:")
for name, weight in zip(results.keys(), model_weights):
    print(f"- {name}: {weight:.3f}")

# 가중 평균 계산
ensemble_val_pred = np.average(ensemble_predictions_val, weights=model_weights, axis=0)
ensemble_test_pred = np.average(ensemble_predictions_test, weights=model_weights, axis=0)

# 앙상블 성능 평가
ensemble_rmse_log = np.sqrt(mean_squared_error(y_val_split, ensemble_val_pred))

# 달러 단위 변환
ensemble_val_pred_dollars = np.expm1(ensemble_val_pred)
val_true_dollars = np.expm1(y_val_split)
ensemble_rmse_dollars = np.sqrt(mean_squared_error(val_true_dollars, ensemble_val_pred_dollars))
ensemble_r2 = r2_score(val_true_dollars, ensemble_val_pred_dollars)

print(f"\n🏆 앙상블 모델 성능:")
print(f"- RMSE (로그): {ensemble_rmse_log:.4f}")
print(f"- RMSE (달러): ${ensemble_rmse_dollars:,.0f}")
print(f"- R² Score: {ensemble_r2:.4f}")

# 최고 개별 모델과 비교
print(f"\n📈 성능 개선:")
best_individual_rmse = min([result['best_val_rmse'] for result in results.values()])
improvement = (best_individual_rmse - ensemble_rmse_log) / best_individual_rmse * 100
print(f"- 최고 개별 모델 대비 {improvement:.2f}% 개선")

# 앙상블 제출 파일 생성
ensemble_test_pred_dollars = np.expm1(ensemble_test_pred)

ensemble_submission = pd.DataFrame({
    'Id': test_df['Id'],
    'SalePrice': ensemble_test_pred_dollars
})

ensemble_filename = 'house_prices_submission_ensemble.csv'
ensemble_submission.to_csv(ensemble_filename, index=False)

print(f"\n📁 앙상블 제출 파일: {ensemble_filename}")
print(f"\n💡 앙상블 예측 통계:")
print(f"- 평균: ${np.mean(ensemble_test_pred_dollars):,.0f}")
print(f"- 중앙값: ${np.median(ensemble_test_pred_dollars):,.0f}")
print(f"- 표준편차: ${np.std(ensemble_test_pred_dollars):,.0f}")

## 10. 학습 요약 및 개선 방향

In [ ]:
# 최종 요약 정보
print("🎉 House Prices 예측 프로젝트 완료!")
print("=" * 60)
print(f"🏆 최고 개별 모델: {best_model_name}")
print(f"🎯 최고 RMSE (로그): {best_rmse:.4f}")
print(f"🎯 최고 RMSE (달러): ${np.expm1(best_rmse):,.0f}")
print(f"🎭 앙상블 RMSE (로그): {ensemble_rmse_log:.4f}")
print(f"🎭 앙상블 RMSE (달러): ${ensemble_rmse_dollars:,.0f}")

print(f"\n📊 프로젝트 통계:")
print(f"- 훈련 샘플: {len(X_train):,}개")
print(f"- 특성 개수: {len(feature_names)}개")
print(f"- 모델 개수: {len(models)}개")
print(f"- 최고 모델 파라미터: {sum(p.numel() for p in best_model.parameters()):,}개")

print(f"\n💡 핵심 인사이트:")
print(f"- 거주 면적이 집값에 가장 큰 영향")
print(f"- 전체 품질과 건축 연도도 중요한 요인")
print(f"- 동네와 집 스타일이 가격에 영향")
print(f"- 딥러닝으로 {ensemble_r2:.1%} R² 점수 달성")

print(f"\n📈 모델별 성능 랭킹:")
sorted_models = sorted(results.items(), key=lambda x: x[1]['best_val_rmse'])
for i, (name, result) in enumerate(sorted_models):
    rmse = result['best_val_rmse']
    rmse_dollars = np.expm1(rmse)
    print(f"{i+1}. {name:>10s}: RMSE ${rmse_dollars:>7,.0f} (로그: {rmse:.4f})")

print(f"\n🚀 개선 방향:")
print(f"1. 🔧 고급 특성 엔지니어링")
print(f"   - 집 크기 비율 특성 (방 크기 대비 전체 면적 등)")
print(f"   - 동네별 평균 가격과의 차이")
print(f"   - 시설물 조합 특성 (수영장 + 차고 등)")

print(f"\n2. 🎯 모델 개선")
print(f"   - Gradient Boosting (XGBoost, LightGBM) 비교")
print(f"   - 교차 검증으로 더 안정적인 평가")
print(f"   - 하이퍼파라미터 최적화 (Optuna, Hyperopt)")

print(f"\n3. 📊 데이터 개선")
print(f"   - 더 정교한 결측치 처리")
print(f"   - 이상치 분석 및 처리 개선")
print(f"   - 범주형 변수의 더 나은 인코딩")

print(f"\n4. 🎭 앙상블 개선")
print(f"   - 스태킹 앙상블")
print(f"   - 다양한 모델 타입 조합")
print(f"   - 동적 가중치 계산")

print(f"\n🔗 다음 단계:")
print(f"1. Kaggle에 제출하여 실제 점수 확인")
print(f"2. 리더보드 분석 및 다른 참가자 솔루션 연구")
print(f"3. 특성 중요도 심화 분석")
print(f"4. 다른 회귀 대회 도전 (Bike Sharing, Sales 등)")
print(f"5. 실제 부동산 데이터로 응용 프로젝트")